In [1]:
from pathlib import Path
import yt_dlp
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
import os

In [2]:
# Load environment variables and API keys
load_dotenv()

True

In [3]:
# Video URL -> will be switched to user input in prod

url = "https://www.youtube.com/watch?v=gJfySWd2HVI"

# Video ID extraction

video_id = url.split("v=")[1].split("&")[0]

with yt_dlp.YoutubeDL({"quiet": True}) as ydl:
    audio_length = ydl.extract_info(url, download=False)["duration"]

if not os.path.exists(f"./transcripts/"):
    os.makedirs(f"./transcripts/")

transcript_path = Path(f"./transcripts/{video_id}.txt")

if not transcript_path.exists():

    try:

        # Trying to find captions using youtube_transcript_api

        print("Trying to find captions using youtube_transcript_api...")
        transcript_list = YouTubeTranscriptApi().list(video_id)
        transcript_obj = transcript_list.find_manually_created_transcript(["en"])

        if transcript_obj:
            transcript = " ".join(snippet.text for snippet in transcript_obj.fetch())
            print(transcript)
        else:
            raise RuntimeError("No transcript found")

    except (TranscriptsDisabled, NoTranscriptFound):

        # Fallback: download audio for faster-whisper transcription

        from faster_whisper import WhisperModel

        print("Generating captions using faster-whisper...")

        whisper_model = WhisperModel("./faster-whisper-large-v3-turbo")

        out_dir = Path(f"./downloads/{video_id}")
        out_dir.mkdir(parents=True, exist_ok=True)

        audio_files = list(out_dir.glob(f"{video_id}.*"))
        if audio_files:
            audio_path = audio_files[0]
            print(f"Using existing audio at {audio_path}")
        else:
            output_template = out_dir / f"{video_id}.%(ext)s"

            ydl_opts = {
                "format": "bestaudio/best",
                "outtmpl": str(output_template),
                "quiet": True,
            }

            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([url])

            audio_path = next(out_dir.glob(f"{video_id}.*"))
            print(f"Downloaded audio to {audio_path}")

        segments, info = whisper_model.transcribe(str(audio_path))
        transcript = ""
        for segment in segments:
            print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))
            transcript += segment.text + " "
        print(transcript)

    # Saving the transcript to a file
    with open(transcript_path, "w") as f:
        f.write(transcript) 

else:
    print(f"Transcript already exists. Moving on...")

Transcript already exists. Moving on...


In [4]:
# Ingestion

# Doc-Loader

from langchain_community.document_loaders import TextLoader

doc_loader = TextLoader(transcript_path)
docs = doc_loader.load()

transcript = docs[0].page_content

transcript

C:\Users\immad\AppData\Local\Temp\ipykernel_6316\313735928.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


" All right, who would, uh, like some yams, Will?  Oh, you'd like that, wouldn't you?  What?  Oh, you know what?  Can we please keep the chicken and the turkey  and everything on the other side of the table?  The smell is just blech.  Typical!  I'm sorry, what?  I said it was typical.  Typical of you.  Rachel Green, Queen Rachel,  does whatever she wants in a little Rachel-land.  Seriously, who is this guy?  Um, I'm sorry, do you have a problem with me?  I don't know. Do I? Do I?  I think you do.  Apparently you were, um, a little mean to him in high school.  A little mean? You made my life miserable.  I'm, I'm, I had no idea. I'm sorry, I...  Well, you should be.  Screw it. Bring on the yams.  Will, but you've worked so hard.  Yams!  Okay.  Uh, Will, I just want to say  that I'm real sorry  for whatever I, I did to you in high school.  Oh, it wasn't just me.  We had a club.  You had a club?  That's right.  The I Hate Rachel Green Club.  Oh, my God.  So what, did you all just join toge

In [7]:
# Chunking -- only if using RAG pipeline

if audio_length > (60 * 10):

    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model_name = "cnmoro/Qwen0.5b-RagSemanticChunker"

    chunkingModel = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto"
    )
    chunkingModel.to(device)
    chunkingTokenizer = AutoTokenizer.from_pretrained(model_name)

    messages = [
        {"role": "system", "content": "Your job is to act as a semantic chunker.\nThe goal is to separate the content into semantically relevant groupings.\nEach group must be delimited by the \"段\" character."},
        {"role": "user", "content": f"Segment this text:\n\n```text\n{transcript}\n```"}
    ]
    chunkingInputs = chunkingTokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = chunkingTokenizer([chunkingInputs], return_tensors="pt").to(chunkingModel.device)

    generated_ids = chunkingModel.generate(
        **model_inputs,
        max_new_tokens=2048
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = chunkingTokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    print(response)

    # only keeping text after ```text

    preprocessed_transcript = response.split("```text")[1]

    chunks = preprocessed_transcript.split("段")

    print(chunks)

    del chunkingModel, chunkingTokenizer, model_inputs, generated_ids, response, preprocessed_transcript
    torch.cuda.empty_cache()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sure, here is the text segmented using the "段" character:

```text
All right, who would, uh, like some yams, Will?  Oh, you'd like that, wouldn't you?  What?  Oh, you know what?  Can we please keep the chicken and the turkey  and everything on the other side of the table?  The smell is just blech.  Typical!  I'm sorry, what?  I said it was typical.  Typical of you.  
段  
Rachel Green, Queen Rachel,  does whatever she wants in a little Rachel-land.  Seriously, who is this guy?  Um, I'm sorry, do you have a problem with me?  I don't know. Do I? Do I?  I think you do.  Apparently you were, um, a little mean to him in high school.  A little mean? You made my life miserable.  I'm, I'm, I had no idea. I'm sorry, I...  Well, you should be.  Screw it.  
段  
Bring on the yams.  Will, but you've worked so hard.  Yams!  Okay.  Uh, Will, I just want to say  that I'm real sorry  for whatever I, I did to you in high school.  Oh, it wasn't just me.  We had a club.  You had a club?  That's right.  The

In [ ]:
# Only creating embeddings if audio is > 10 mins

if audio_length > (60 * 10):

    from langchain_chroma import Chroma
    from langchain_huggingface import HuggingFaceEmbeddings

    # Requires transformers>=4.51.0
    # Requires sentence-transformers>=2.7.0

    docs = [Document(page_content=chunk) for chunk in chunks]

    # Load the model
    embeddingModel = "Qwen/Qwen3-Embedding-0.6B"

    embeddingFunction = HuggingFaceEmbeddings(
        model_name=embeddingModel
    )

    chroma_client = Chroma(
        collection_name="youtube-transcripts",
        embedding_function=embeddingFunction,
        persist_directory="./chroma_db"
    )

    chroma_client.add_documents(docs)
    print(f"Indexed {len(docs)} chunks into Chroma")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Indexed 5 chunks into Chroma


In [9]:
# Model and Tokenizer Init

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
from dotenv import load_dotenv
import torch
import os

CAG_SYSTEM_PROMPT = "Answer questions based on the provided video transcript."
CAG_CACHE_VERSION = 1  # bump when prefill format changes to invalidate stale caches

if not os.path.exists(f"KV_Caches/"):
    os.makedirs(f"KV_Caches/")

if not os.path.exists(f"Transcript_Lengths/"):
    os.makedirs(f"Transcript_Lengths/")

load_dotenv()

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

queryModel = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-3B-Instruct",
    quantization_config=quantization_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

W0629 10:12:55.770000 6316 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [ ]:
# CAG pipeline

cache_path = Path(f"KV_Caches/{video_id}.pt")
transcriptLength_path = Path(f"Transcript_Lengths/{video_id}_length.pt")
cacheVersion_path = Path(f"Transcript_Lengths/{video_id}_cache_version.pt")

if audio_length <= 60*10:

    cache_valid = (
        cache_path.exists()
        and transcriptLength_path.exists()
        and cacheVersion_path.exists()
        and torch.load(cacheVersion_path, weights_only=False) == CAG_CACHE_VERSION
    )

    if cache_valid:
        kv_cache = torch.load(cache_path, weights_only=False, map_location=device)
        prefix_length = torch.load(transcriptLength_path, weights_only=False)
        print(f"Loaded KV cache ({prefix_length} prefix tokens)")

    else:
        with open(transcript_path, "r") as f:
            transcript_text = f.read()

        prefill_messages = [
            {
                "role": "system",
                "content": f"{CAG_SYSTEM_PROMPT}\n\nTranscript:\n{transcript_text}",
            }
        ]
        prefill_inputs = tokenizer.apply_chat_template(
            prefill_messages,
            return_tensors="pt",
            add_generation_prompt=False,
        ).to(device)

        prefix_length = prefill_inputs.input_ids.shape[1]

        with torch.no_grad():
            outputs = queryModel(**prefill_inputs, use_cache=True)

        kv_cache = outputs.past_key_values
        if hasattr(kv_cache, "to_legacy_cache"):
            kv_cache = kv_cache.to_legacy_cache()

        torch.save(kv_cache, cache_path)
        torch.save(prefix_length, transcriptLength_path)
        torch.save(CAG_CACHE_VERSION, cacheVersion_path)
        print(f"Built and saved KV cache ({prefix_length} prefix tokens)")

C:\Users\immad\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Built and saved KV cache (1226 prefix tokens)


In [ ]:
# Query Time

if audio_length <= (60 * 10):

    cache_path = Path(f"KV_Caches/{video_id}.pt")
    transcriptLength_path = Path(f"Transcript_Lengths/{video_id}_length.pt")

    query = "What did Rachel do?"

    with open(transcript_path, "r") as f:
        transcript_text = f.read()

    query_messages = [
        {
            "role": "system",
            "content": f"{CAG_SYSTEM_PROMPT}\n\nTranscript:\n{transcript_text}",
        },
        {"role": "user", "content": query},
    ]
    full_inputs = tokenizer.apply_chat_template(
        query_messages,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(device)

    prefix_length = torch.load(transcriptLength_path, weights_only=False)
    new_input_ids = full_inputs.input_ids[:, prefix_length:]

    kv_cache = torch.load(cache_path, weights_only=False, map_location=device)
    if isinstance(kv_cache, tuple):
        kv_cache = DynamicCache.from_legacy_cache(kv_cache)

    with torch.no_grad():
        output = queryModel.generate(
            input_ids=new_input_ids,
            attention_mask=full_inputs.attention_mask,
            past_key_values=kv_cache,
            max_new_tokens=256,
            do_sample=False,
        )

    print(tokenizer.decode(output[0], skip_special_tokens=True))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


user

What did Rachel do?assistant

According to the transcript, Rachel Green was the subject of bullying and teasing in high school, specifically by a group of people who formed the "I Hate Rachel Green Club" with Ross. Rachel had no idea about this group and was unaware of the extent of the bullying she had endured.


In [10]:
# RAG pipeline

# if audio_length > (60 * 10):

query = "What did Rachel do?"

retrieved = chroma_client.similarity_search(query, k=3)

context = "\n\n".join(doc.page_content for doc in retrieved)

formal_query = f"Respond to the following text: \n {query} \n {context}"

messages = [
    {"role": "system", "content": "Answer questions based on the provided video transcript."},
    {"role": "user", "content": formal_query},
]
formatted_formal_query = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

tokenized_formal_query = tokenizer(formatted_formal_query, return_tensors="pt").to(device)

with torch.no_grad():
    generated = queryModel.generate(
        **tokenized_formal_query,
        max_new_tokens=256,
    )

response = tokenizer.decode(generated[0], skip_special_tokens=True)

print(response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
C:\Users\immad\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


system

Cutting Knowledge Date: December 2023
Today Date: 29 Jun 2026

Answer questions based on the provided video transcript.user

Respond to the following text: 
 What did Rachel do? 
   
Rachel Green, Queen Rachel,  does whatever she wants in a little Rachel-land.  Seriously, who is this guy?  Um, I'm sorry, do you have a problem with me?  I don't know. Do I? Do I?  I think you do.  Apparently you were, um, a little mean to him in high school.  A little mean? You made my life miserable.  I'm, I'm, I had no idea. I'm sorry, I...  Well, you should be.  Screw it.  


  
Bring on the yams.  Will, but you've worked so hard.  Yams!  Okay.  Uh, Will, I just want to say  that I'm real sorry  for whatever I, I did to you in high school.  Oh, it wasn't just me.  We had a club.  You had a club?  That's right.  The I Hate Rachel Green Club.  Oh, my God.  So what, did you all just join together to hate me?  Who else was in this club?  Me and Ross.  No need to point.  She knows who Ross is.  I h